# Task 1

This section loads the NSW region summary dataset and prepares a cleaned version for analysis. The main issue in this file is that many indicators are only available in selected years, so missing values should not be filled with zero. Instead, the workflow keeps the original values, standardises the column names, extracts the unit from each indicator description, counts how many valid years each indicator has, and reshapes the dataset into a long format for later statistics and visualisation.

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

current_dir = Path.cwd()
data_path = current_dir / "data" / "Region summary_ New South Wales STE 1.csv"

df = pd.read_csv(data_path)
display(df.head())


## Cleaning logic

The raw table is a wide table, which means each row is one indicator and each year is stored in a separate column. To make the data easier to analyse, the cleaning step below does three things. First, it standardises the text column names and year column names. Second, it separates the measurement unit from the description column and records how many non-null years each indicator has. Third, it reshapes the table into a long format so that each row represents one indicator-year value.

In [ ]:
task1_df = df.rename(
    columns={
        "Measure Code": "measure_code",
        "Parent Description": "parent_description",
        "Description": "description",
    }
).copy()

task1_df.columns = [f"y_{col}" if str(col).isdigit() else col for col in task1_df.columns]
year_cols = [col for col in task1_df.columns if col.startswith("y_")]

task1_df["unit"] = task1_df["description"].str.extract(r"\(([^()]*)\)\s*$")
task1_df["description_clean"] = task1_df["description"].str.replace(r"\s*\([^()]*\)\s*$", "", regex=True)
task1_df["non_null_years"] = task1_df[year_cols].notna().sum(axis=1)
task1_df = task1_df[task1_df["non_null_years"] > 0].copy()

task1_long_df = task1_df.melt(
    id_vars=["measure_code", "parent_description", "description", "description_clean", "unit", "non_null_years"],
    value_vars=year_cols,
    var_name="year",
    value_name="value",
)

task1_long_df = task1_long_df.dropna(subset=["value"]).copy()
task1_long_df["year"] = task1_long_df["year"].str.replace("y_", "").astype(int)
task1_analysis_df = task1_long_df[task1_long_df["year"] != 2025].copy()

cleaning_summary_df = pd.DataFrame(
    {
        "item": [
            "original_rows",
            "rows_after_drop_all_null_years",
            "long_table_rows",
            "analysis_rows_without_2025",
            "rows_with_only_one_non_null_year",
            "non_null_values_in_2025",
        ],
        "value": [
            len(df),
            len(task1_df),
            len(task1_long_df),
            len(task1_analysis_df),
            int((task1_df["non_null_years"] == 1).sum()),
            int(task1_long_df[task1_long_df["year"] == 2025].shape[0]),
        ],
    }
)

display(cleaning_summary_df)
display(task1_df.head())
display(task1_analysis_df.head())


# Task 2

This section builds the geographic dataset for four selected areas. Each area is linked to one SA4 zone. For each selected SA4, the workflow first uses the NSW Administrative Boundaries API to find the SA2 regions contained inside that SA4. It then uses the NSW POI API to collect the POIs inside the bounding box of each SA2. Finally, the results are combined and saved to a local SQLite database.

## Selected SA4 codes

The group uses four SA4 codes: `120`, `117`, `118`, and `125`. The setup cell below matches each code to its official SA4 name and defines a small helper function so the same logic can be reused for each selected area.

In [ ]:
import importlib
import task2_sa2

importlib.reload(task2_sa2)

available_sa4s = task2_sa2.list_nsw_sa4s()
area_zone_df = pd.DataFrame(
    {
        "area_name": ["area_1", "area_2", "area_3", "area_4"],
        "sa4_code": ["120", "117", "118", "125"],
    }
)

area_zone_df = area_zone_df.merge(
    available_sa4s.rename(columns={"SA4_CODE": "sa4_code", "SA4_NAME": "sa4_name"}),
    on="sa4_code",
    how="left",
)

def run_area_zone(area_name, sa4_code):
    sa2_bbox_df = task2_sa2.get_sa2_bbox_df_by_code(sa4_code)
    poi_df = task2_sa2.get_sa4_poi_df_by_code(sa4_code)
    sa2_bbox_df["area_name"] = area_name
    poi_df["area_name"] = area_name
    poi_count_df = poi_df.groupby("sa2_name").size().reset_index(name="poi_count")
    return sa2_bbox_df, poi_df, poi_count_df.sort_values("poi_count", ascending=False)

display(area_zone_df)


## Area 1

Area 1 is linked to `SA4_CODE = 120`, which corresponds to `Sydney - Inner West`. The cell below returns three tables: the SA2 regions contained in this SA4, a sample of the POI rows collected for this zone, and a count of how many POIs were assigned to each SA2.

In [ ]:
area_1_sa2_bbox_df, area_1_poi_df, area_1_poi_count_df = run_area_zone("area_1", "120")
display(area_1_sa2_bbox_df[["area_name", "sa4_code", "sa4_name", "sa2_main", "sa2_name"]])
display(area_1_poi_df.head())
display(area_1_poi_count_df)


## Area 2

Area 2 is linked to `SA4_CODE = 117`, which corresponds to `Sydney - City and Inner South`. The same process is repeated here so that the results remain separate by area.

In [ ]:
area_2_sa2_bbox_df, area_2_poi_df, area_2_poi_count_df = run_area_zone("area_2", "117")
display(area_2_sa2_bbox_df[["area_name", "sa4_code", "sa4_name", "sa2_main", "sa2_name"]])
display(area_2_poi_df.head())
display(area_2_poi_count_df)


## Area 3

Area 3 is linked to `SA4_CODE = 118`, which corresponds to `Sydney - Eastern Suburbs`. Keeping the outputs separate makes it easier to show which SA2 and POI records belong to each selected area.

In [ ]:
area_3_sa2_bbox_df, area_3_poi_df, area_3_poi_count_df = run_area_zone("area_3", "118")
display(area_3_sa2_bbox_df[["area_name", "sa4_code", "sa4_name", "sa2_main", "sa2_name"]])
display(area_3_poi_df.head())
display(area_3_poi_count_df)


## Area 4

Area 4 is linked to `SA4_CODE = 125`, which corresponds to `Sydney - Parramatta`. This final section follows the same structure so that all four areas can be combined consistently later.

In [ ]:
area_4_sa2_bbox_df, area_4_poi_df, area_4_poi_count_df = run_area_zone("area_4", "125")
display(area_4_sa2_bbox_df[["area_name", "sa4_code", "sa4_name", "sa2_main", "sa2_name"]])
display(area_4_poi_df.head())
display(area_4_poi_count_df)


## Combine and save

After each area's SA2 and POI data has been collected, the four outputs are concatenated into two full tables: one SA2 table and one POI table. These tables are then saved into a local SQLite database. The database links `poi` to `sa2_bbox` through the `sa2_main` field, which makes it possible to run joined queries later.

In [ ]:
all_sa2_bbox_df = pd.concat(
    [area_1_sa2_bbox_df, area_2_sa2_bbox_df, area_3_sa2_bbox_df, area_4_sa2_bbox_df],
    ignore_index=True,
)

all_poi_df = pd.concat(
    [area_1_poi_df, area_2_poi_df, area_3_poi_df, area_4_poi_df],
    ignore_index=True,
)

all_sa2_bbox_df = all_sa2_bbox_df[
    ["area_name", "sa4_code", "sa4_name", "sa2_main", "sa2_name", "state_name", "area_sqkm", "bbox_xmin", "bbox_ymin", "bbox_xmax", "bbox_ymax"]
]
all_poi_df = all_poi_df[
    ["objectid", "poiname", "poitype", "poigroup", "longitude", "latitude", "area_name", "sa4_code", "sa4_name", "sa2_main", "sa2_name"]
]

db_path = current_dir / "data" / "task2_poi.db"
task2_sa2.save_to_sqlite(all_sa2_bbox_df, all_poi_df, str(db_path))

schema_df = task2_sa2.read_sql(
    "select name, sql from sqlite_master where type = 'table' order by name",
    str(db_path),
)
join_df = task2_sa2.read_sql(
    "select p.poi_id, p.area_name, p.poiname, p.poitype, s.sa4_code, s.sa4_name, s.sa2_name from poi p join sa2_bbox s on p.sa2_main = s.sa2_main limit 20",
    str(db_path),
)

display(schema_df)
display(join_df)
